# 📊 01 - Sample Data Generation

**Run this first to generate realistic ticket data** - 

## What this notebook does:
- ✅ Drops and recreates the Unity Catalog schema for a clean start
- ✅ Generates **completely unstructured, realistic ticket data**
- ✅ Creates messy, natural language descriptions perfect for LLM extraction
- ✅ Saves data to Unity Catalog Delta tables
- ✅ Ready for next notebook: `02_action_extraction.ipynb`

**Prerequisites:** None - this is the starting point

## Data Quality: 100% unstructured and realistic!


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets


In [2]:
# Clean start - Drop and recreate schema
print("🗑️ Step 1: Dropping existing schema for clean start...")

try:
    spark.sql(f"DROP SCHEMA IF EXISTS {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']} CASCADE")
    print("✅ Existing schema dropped")
except Exception as e:
    print(f"ℹ️ Schema drop result: {e}")

# Create fresh schema
print("🏗️ Step 2: Creating fresh Unity Catalog schema...")
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}
COMMENT 'Schema for Databricks ticket classification system using Unity Catalog'
""")

print("✅ Fresh schema created")
print(f"🎯 Schema: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


🗑️ Step 1: Dropping existing schema for clean start...
✅ Existing schema dropped
🏗️ Step 2: Creating fresh Unity Catalog schema...
✅ Fresh schema created
🎯 Schema: quickstart_catalog_vkm_external.classify_tickets


In [3]:
# 📈 Generate 7-day time series data for AI forecasting
print("📝 Step 3: Generating 7-day time series data for AI forecasting...")

from datetime import datetime, timedelta
import random

# Generate 7 days of data with realistic patterns
tickets_data = []
base_date = datetime.now() - timedelta(days=7)

# Realistic ticket descriptions with more variety
ticket_descriptions = [
    "hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.",
    "urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.",
    "i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?",
    "our monitoring alerts are going crazy. everything shows red but the app seems to be working fine. not sure if it's a false positive or if something is actually broken. help?",
    "the deployment failed again and now production is down. we rolled back but need to figure out what went wrong. this is the third time this month. we need better testing.",
    "customers are reporting that their data is missing after the last update. this is a big problem and we need to investigate immediately. could be a data migration issue.",
    "the new security patch broke our authentication. users can't log in and we're getting flooded with support tickets. need to fix this before the security team gets involved.",
    "our cloud costs are through the roof this month. something is using way more resources than usual. need to find out what's causing the spike and optimize it.",
    "the backup system isn't working and we haven't had a successful backup in 3 days. this is a major risk and we need to fix it before something bad happens.",
    "the new microservice is causing memory leaks and crashing the whole system. we need to either fix it or disable it until we can figure out what's wrong.",
    "database connection timeouts are happening randomly throughout the day. this is affecting our reporting dashboard and users are getting frustrated.",
    "the new user interface is causing browser crashes on older devices. we need to either fix the compatibility or provide a fallback version.",
    "our email notifications stopped working after the last update. customers aren't receiving order confirmations and they're calling support.",
    "the payment gateway is intermittently failing. some transactions go through while others fail with no clear pattern. this is costing us money.",
    "mobile app is crashing on iOS devices after the latest update. android users are fine but we're losing iOS customers.",
    "the search functionality is returning irrelevant results. users can't find what they're looking for and it's affecting sales.",
    "our analytics dashboard is showing incorrect data. the numbers don't match what we see in the database and it's confusing management.",
    "the file upload feature is broken. users can't attach documents and it's blocking several business processes.",
    "the chat support widget disappeared from our website. customers can't reach support and they're complaining on social media.",
    "the password reset functionality is not working. users are locked out of their accounts and can't access their data."
]

# Generate data for each day with realistic patterns
for day in range(7):
    current_date = base_date + timedelta(days=day)
    
    # Create realistic daily patterns:
    # - More tickets on weekdays (Mon-Fri)
    # - Fewer tickets on weekends
    # - Higher priority tickets on certain days
    if current_date.weekday() < 5:  # Weekday
        daily_ticket_count = random.randint(8, 15)  # More tickets on weekdays
        urgent_probability = 0.3  # 30% chance of urgent tickets
    else:  # Weekend
        daily_ticket_count = random.randint(3, 8)   # Fewer tickets on weekends
        urgent_probability = 0.5  # 50% chance of urgent tickets (weekend issues are often urgent)
    
    # Generate tickets for this day
    for ticket_num in range(daily_ticket_count):
        ticket_id = f"TICKET_{day+1:02d}_{ticket_num+1:03d}"
        
        # Select random description
        description = random.choice(ticket_descriptions)
        
        # Create realistic time patterns within the day
        # More tickets in morning (9-11 AM) and afternoon (2-4 PM)
        hour = random.choices(
            [9, 10, 11, 14, 15, 16, 17, 18, 19, 20, 21, 22],
            weights=[3, 4, 3, 2, 3, 4, 3, 2, 1, 1, 1, 1]
        )[0]
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        
        created_timestamp = current_date.replace(hour=hour, minute=minute, second=second)
        
        # Determine priority based on day and random factors
        if random.random() < urgent_probability:
            priority = random.choice(["1 - Critical", "2 - High"])
        else:
            priority = random.choice(["2 - High", "3 - Medium", "4 - Low"])
        
        # Random assignment details
        assigned_to = random.choice(["john.smith@company.com", "sarah.jones@company.com", "mike.wilson@company.com", "lisa.brown@company.com", "unassigned"])
        assignment_group = random.choice(["Platform Team", "Security Team", "Database Team", "Frontend Team", "DevOps Team", "Unassigned"])
        requested_by = random.choice(["customer@example.com", "internal.user@company.com", "support@company.com", "manager@company.com"])
        state = random.choice(["New", "In Progress", "Assigned", "Pending"])
        
        tickets_data.append({
            "ticket_id": ticket_id,
            "short_description": f"Issue #{day+1}-{ticket_num+1} - {random.choice(['Urgent', 'Critical', 'Important', 'Help needed', 'Problem'])}",
            "description": description,
            "assigned_to": assigned_to,
            "assignment_group": assignment_group,
            "requested_by": requested_by,
            "priority": priority,
            "state": state,
            "created_date": created_timestamp,
            "created_timestamp": created_timestamp
        })

print(f"✅ Generated {len(tickets_data)} tickets over 7 days")
print("📊 Daily ticket distribution:")
daily_counts = {}
for ticket in tickets_data:
    date_str = ticket['created_date'].strftime('%Y-%m-%d')
    daily_counts[date_str] = daily_counts.get(date_str, 0) + 1

for date_str in sorted(daily_counts.keys()):
    print(f"   {date_str}: {daily_counts[date_str]} tickets")

print("\n📋 Sample tickets from different days:")
for i, ticket in enumerate(tickets_data[::len(tickets_data)//5]):  # Show 5 samples
    print(f"\n{i+1}. {ticket['ticket_id']}: {ticket['short_description']}")
    print(f"   Created: {ticket['created_date']}")
    print(f"   Priority: {ticket['priority']}")
    print(f"   Description: {ticket['description'][:80]}...")


📝 Step 3: Generating 7-day time series data for AI forecasting...
✅ Generated 64 tickets over 7 days
📊 Daily ticket distribution:
   2025-09-13: 5 tickets
   2025-09-14: 7 tickets
   2025-09-15: 9 tickets
   2025-09-16: 12 tickets
   2025-09-17: 10 tickets
   2025-09-18: 12 tickets
   2025-09-19: 9 tickets

📋 Sample tickets from different days:

1. TICKET_01_001: Issue #1-1 - Important
   Created: 2025-09-13 15:12:58.161785
   Priority: 1 - Critical
   Description: customers are reporting that their data is missing after the last update. this i...

2. TICKET_03_001: Issue #3-1 - Urgent
   Created: 2025-09-15 14:30:33.161785
   Priority: 2 - High
   Description: database connection timeouts are happening randomly throughout the day. this is ...

3. TICKET_04_004: Issue #4-4 - Critical
   Created: 2025-09-16 10:04:16.161785
   Priority: 1 - Critical
   Description: i need help with the new feature we're building. the api is returning weird erro...

4. TICKET_05_004: Issue #5-4 - Critical

In [4]:
# Create DataFrame and save to Unity Catalog
print("💾 Step 4: Saving 7-day time series data to Unity Catalog...")

# Create DataFrame from 7-day ticket data
df_tickets = spark.createDataFrame(tickets_data)

# Save to Unity Catalog Delta table
df_tickets.write.format("delta").mode("overwrite").saveAsTable(TABLES["raw_tickets"])

print("✅ 7-day time series data saved to Unity Catalog Delta table:")
print(f"🎯 Table: {TABLES['raw_tickets']}")

# Display sample data with timestamps
print("\n📊 Sample 7-day data from Unity Catalog:")
display(df_tickets.select("ticket_id", "short_description", "created_date", "priority").limit(5))

# Show time series summary statistics
print(f"\n📈 Time Series Summary Statistics:")
print(f"   Total tickets generated: {df_tickets.count()}")
print(f"   Date range: {df_tickets.select(min('created_date')).collect()[0][0]} to {df_tickets.select(max('created_date')).collect()[0][0]}")
print(f"   Unique assignment groups: {df_tickets.select('assignment_group').distinct().count()}")

# Show daily ticket counts
print(f"\n📅 Daily Ticket Counts:")
daily_summary = df_tickets.groupBy(date_format('created_date', 'yyyy-MM-dd').alias('date')).count().orderBy('date')
daily_summary.show()

# Show priority distribution over time
print(f"\n📊 Priority Distribution by Day:")
priority_by_day = df_tickets.groupBy(
    date_format('created_date', 'yyyy-MM-dd').alias('date'),
    'priority'
).count().orderBy('date', 'priority')
priority_by_day.show()

print("\n🎯 Ready for AI forecasting with 7 days of realistic time series data!")
print("📈 Next: Run the AI forecasting SQL queries to see predictions!")


💾 Step 4: Saving 7-day time series data to Unity Catalog...


✅ 7-day time series data saved to Unity Catalog Delta table:
🎯 Table: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📊 Sample 7-day data from Unity Catalog:


,ticket_id,short_description,created_date,priority
0,TICKET_01_001,Issue #1-1 - Important,2025-09-13 19:12:58.161785,1 - Critical
1,TICKET_01_002,Issue #1-2 - Urgent,2025-09-13 20:26:11.161785,2 - High
2,TICKET_01_003,Issue #1-3 - Problem,2025-09-13 15:44:33.161785,2 - High
3,TICKET_01_004,Issue #1-4 - Important,2025-09-13 14:15:46.161785,1 - Critical
4,TICKET_01_005,Issue #1-5 - Help needed,2025-09-13 13:27:38.161785,2 - High



📈 Time Series Summary Statistics:


   Total tickets generated: 64


   Date range: 2025-09-13 09:27:38.161785 to 2025-09-19 21:14:20.161785


   Unique assignment groups: 6

📅 Daily Ticket Counts:


+----------+-----+
|      date|count|
+----------+-----+
|2025-09-13|    5|
|2025-09-14|    7|
|2025-09-15|    7|
|2025-09-16|   13|
|2025-09-17|    9|
|2025-09-18|   13|
|2025-09-19|    9|
|2025-09-20|    1|
+----------+-----+


📊 Priority Distribution by Day:


+----------+------------+-----+
|      date|    priority|count|
+----------+------------+-----+
|2025-09-13|1 - Critical|    2|
|2025-09-13|    2 - High|    3|
|2025-09-14|1 - Critical|    2|
|2025-09-14|    2 - High|    2|
|2025-09-14|     4 - Low|    3|
|2025-09-15|1 - Critical|    2|
|2025-09-15|    2 - High|    3|
|2025-09-15|  3 - Medium|    1|
|2025-09-15|     4 - Low|    1|
|2025-09-16|1 - Critical|    1|
|2025-09-16|    2 - High|    4|
|2025-09-16|  3 - Medium|    5|
|2025-09-16|     4 - Low|    3|
|2025-09-17|    2 - High|    5|
|2025-09-17|  3 - Medium|    4|
|2025-09-18|1 - Critical|    4|
|2025-09-18|    2 - High|    6|
|2025-09-18|  3 - Medium|    1|
|2025-09-18|     4 - Low|    2|
|2025-09-19|1 - Critical|    1|
+----------+------------+-----+
only showing top 20 rows

🎯 Ready for AI forecasting with 7 days of realistic time series data!
📈 Next: Run the AI forecasting SQL queries to see predictions!
